# Epoch averaging — BCI Competition III, Dataset II (Subject B)

Адаптация `epoch_averaging.ipynb` под новый датасет (BCI Competition III, dataset II,
P300-speller), извлечение которого показано в `notebooks/time_series_tests.ipynb`.

Отличия от исходного ноутбука:
- Данные берутся из `matrix_dataset` (Google Drive) через `P300Getter` (Subject B).
- Используется **штатное train/test-разбиение** датасета: `Subject_B_Train.mat` → train,
  `Subject_B_Test.mat` → val (метки теста берутся из известной строки символов).
- Каждая эпоха сводится к одному электроду (`Pz`) → 1D-ряд из 72 отсчётов,
  что является прямым аналогом single-channel PZ из multi_eeg датасета.
- В датасете всего два субъекта, поэтому межсубъектные стратегии
  (cross-subject / mixed / K-trials) и time-shift опущены. Берём **только Subject B**.
- Гиперпараметры и базовые эксперименты — те же, что в оригинале
  (epoch averaging N=5, N=10 для EEGNet / BaseCNN / SVM, MC vs SC).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/pattern_recognition")

# Colab's kernel is Python 3.12 and already ships torch / numpy / scipy / pandas /
# sklearn / matplotlib / seaborn. Poetry would build a *separate* py3.10 venv whose
# compiled extensions can't be imported by the 3.12 kernel, so install the
# missing deps straight into the running kernel instead.
!pip install -q mne torch_geometric captum visualtorch torchviz statsmodels openpyxl onnx onnxscript

In [ ]:
sys.path.append('/content/drive/MyDrive/pattern_recognition/src')

from data import GraphMatrixDataset, CNNMatrixDataset
from utils import P300Getter, train_model, plot_sample, show_progress, validate_model, infer_model
from interpretation import *
from models_cnn import *
from models_gnn import *
from graph import get_delaunay_graph, get_pos_init_graph, plot_graph, get_neighbors_graph

In [ ]:
import math
import glob

import mne
import pandas as pd
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
from torch import nn, optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader
from torch.nn import functional as F

import time
from sklearn.model_selection import train_test_split
from statsmodels.stats.proportion import proportion_confint

## Вспомогательные функции

In [ ]:
def standardize_per_sample(X, eps=1e-8):
    """
    Standardize each sample (row) of X to zero mean and unit variance.
    X: array-like of shape (n_samples, n_timestamps)
    """
    X = np.asarray(X, dtype=np.float32)
    mean = X.mean(axis=1, keepdims=True)
    std = X.std(axis=1, keepdims=True)  # population std (ddof=0)
    return (X - mean) / np.maximum(std, eps)

In [ ]:
@torch.no_grad()
def borderline_smote_torch(
    X: torch.Tensor,
    y: torch.Tensor,
    minority_label: int = 1,
    k: int = 5,          # minority neighbors for synthesis
    m: int = 10,         # neighbors for "danger" detection
    ratio: float = 1.0,  # desired minority after oversampling
    generator: torch.Generator | None = None,
    standardize_for_neighbors: bool = True,
    eps: float = 1e-6
):
    """
    Borderline-SMOTE (variant 1) implemented in torch.
    Returns (X_aug, y_aug).
    """
    device = X.device
    dtype = X.dtype

    y = y.view(-1)
    n, d = X.shape

    y_bool_min = (y == minority_label)
    idx_min = torch.nonzero(y_bool_min, as_tuple=False).view(-1)
    idx_maj = torch.nonzero(~y_bool_min, as_tuple=False).view(-1)

    n_min = idx_min.numel()
    n_maj = idx_maj.numel()

    if n_min == 0 or n_maj == 0:
        return X, y

    target_n_min = math.ceil(ratio * n_maj)
    num_new = max(0, target_n_min - n_min)
    if num_new == 0:
        return X, y

    if standardize_for_neighbors:
        mean = X.mean(dim=0, keepdim=True)
        std = X.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
        X_std = (X - mean) / std
    else:
        X_std = X

    X_min = X[idx_min]
    X_min_std = X_std[idx_min]

    m_neighbors = min(m, max(1, n - 1))
    dist_all = torch.cdist(X_min_std, X_std, p=2)
    topk_k = min(m_neighbors + 1, n)
    topk_vals, topk_idx = torch.topk(dist_all, k=topk_k, largest=False, dim=1)

    neigh_idx_list = []
    for row in range(n_min):
        row_idx = topk_idx[row]
        self_idx = idx_min[row]
        mask = row_idx != self_idx
        filtered = row_idx[mask]
        neigh_idx_list.append(filtered[:m_neighbors])

    neigh_idx = torch.stack(neigh_idx_list, dim=0)

    neighbor_labels = y[neigh_idx]
    maj_counts = (neighbor_labels != minority_label).sum(dim=1)

    half = math.ceil(m_neighbors / 2)
    danger_mask = (maj_counts >= half) & (maj_counts < m_neighbors)
    idx_danger_min = idx_min[danger_mask]

    if idx_danger_min.numel() == 0:
        idx_danger_min = idx_min

    X_danger_std = X_std[idx_danger_min]
    dist_min = torch.cdist(X_danger_std, X_min_std, p=2)
    k_neighbors = min(k, max(1, n_min - 1))
    topk_vals_min, topk_idx_min = torch.topk(dist_min, k=min(k_neighbors + 1, n_min),
                                             largest=False, dim=1)
    neigh_min_idx_list = []
    for row in range(idx_danger_min.numel()):
        row_idx = topk_idx_min[row]
        global_min_indices = idx_min[row_idx]
        self_global = idx_danger_min[row]
        mask = global_min_indices != self_global
        filtered = global_min_indices[mask]
        neigh_min_idx_list.append(filtered[:k_neighbors])

    neigh_min_idx = torch.stack(neigh_min_idx_list, dim=0)

    n_danger = idx_danger_min.numel()
    base = num_new // n_danger
    rem = num_new - base * n_danger
    counts = torch.full((n_danger,), base, dtype=torch.long, device=device)
    if rem > 0:
        if generator is None:
            sel = torch.randperm(n_danger, device=device)[:rem]
        else:
            sel = torch.randperm(n_danger, generator=generator, device=device)[:rem]
        counts[sel] += 1

    synth_list = []
    def rand_uniform(shape):
        if generator is None:
            return torch.rand(shape, device=device, dtype=dtype)
        else:
            return torch.rand(shape, generator=generator, device=device, dtype=dtype)

    for row in range(n_danger):
        c = counts[row].item()
        if c <= 0:
            continue
        xi = X[idx_danger_min[row]]
        neighbors_global = neigh_min_idx[row]
        if neighbors_global.numel() == 0:
            continue
        if generator is None:
            choice = torch.randint(low=0, high=neighbors_global.numel(), size=(c,), device=device)
        else:
            choice = torch.randint(low=0, high=neighbors_global.numel(), size=(c,), device=device, generator=generator)
        xj = X[neighbors_global[choice]]
        r = rand_uniform((c, 1))
        xi_rep = xi.unsqueeze(0).expand(c, -1)
        x_new = xi_rep + r * (xj - xi_rep)
        synth_list.append(x_new)

    if len(synth_list) == 0:
        return X, y

    X_syn = torch.cat(synth_list, dim=0).to(device=device, dtype=dtype)
    y_syn = torch.full((X_syn.size(0),), fill_value=minority_label, dtype=y.dtype, device=device)

    X_aug = torch.cat([X, X_syn], dim=0)
    y_aug = torch.cat([y, y_syn], dim=0)
    return X_aug, y_aug

In [ ]:
def build_multichannel_subject_dataset_unique(data, labels, n_channels, seed=None):
    """
    Группирует одноканальные эпохи в наборы по n_channels без перекрытий.
    data: Tensor [T, features], labels: Tensor [T]
    return: X [T', n_channels, features], y [T']
    """
    if seed is not None:
        torch.manual_seed(seed)

    X_out = []
    y_out = []

    for cls in [1, 0]:
        idx = torch.where(labels == cls)[0]
        X_cls = data[idx]

        M = len(X_cls)
        usable = (M // n_channels) * n_channels
        if usable == 0:
            raise ValueError(
                f"Not enough samples for class {cls}: {M} < {n_channels}"
            )

        perm = torch.randperm(M)
        X_cls = X_cls[perm][:usable]

        chunks = X_cls.view(usable // n_channels, n_channels, *X_cls.shape[1:])

        X_out.append(chunks)
        y_out.append(torch.full((usable // n_channels,), cls, dtype=labels.dtype))

    X_out = torch.cat(X_out, dim=0)
    y_out = torch.cat(y_out, dim=0)

    return X_out, y_out

In [ ]:
def multichannel_to_single_channel(X):
    """
    X: Tensor [T, N, features] -> [T, 1, features] (усреднение по эпохам/каналам).
    """
    return X.mean(dim=1, keepdim=True)

## Загрузка нового датасета (BCI Competition III, Subject B)

`P300Getter` извлекает окна (72 отсчёта) вокруг каждой вспышки для каждого из 64 электродов.
Train-метки берутся напрямую из `StimulusType`; test-метки восстанавливаются по известной
строке целевых символов (`test_B_chars`). Затем каждая эпоха сводится к одному электроду
(`Pz`) — прямой аналог single-channel PZ из multi_eeg датасета.

In [ ]:
# Папка с Subject_B_Train.mat / Subject_B_Test.mat и монтажом eloc64.loc на Google Drive.
MATRIX_PATH = '/content/drive/MyDrive/pattern_recognition/matrix_dataset/'
ELECTRODE = 'Pz'  # одиночный электрод (аналог PZ из multi_eeg)

# Известная строка целевых символов для test-части Subject B (метки теста скрыты в .mat).
# Источник: истинные тестовые метки BCI Competition III, Dataset II (Wadsworth), 100 символов.
# ВАЖНО: при ошибке в строке все val-метрики Subject B будут некорректны — сверьте при первом запуске.
test_B_chars = list('MERMIROOMUHJPXJOHUVLEORZP3GLOO7AUFDKEFTWEOOALZOP9ROCGZET1Y19EWX65QUYU7NAK_4YCJDVDNGQXODBEV2B5EFDIDNR')
assert len(test_B_chars) == 100, f'ожидалось 100 символов, получено {len(test_B_chars)}'

eloc = mne.channels.read_custom_montage(MATRIX_PATH + 'eloc64.loc')

# Индекс выбранного электрода (имена в монтаже имеют хвостовые точки, напр. 'Pz..')
ch_norm = [c.lower().strip('.') for c in eloc.ch_names]
ch_idx = ch_norm.index(ELECTRODE.lower())

train_B_raw = scipy.io.loadmat(MATRIX_PATH + 'Subject_B_Train.mat')
test_B_raw = scipy.io.loadmat(MATRIX_PATH + 'Subject_B_Test.mat')

B_train_ds = P300Getter(train_B_raw, eloc, sample_size=72)
B_test_ds = P300Getter(test_B_raw, eloc, sample_size=72, target_chars=test_B_chars)

B_train_ds.get_cnn_p300_dataset(filter=True)
B_test_ds.get_cnn_p300_dataset(filter=True)

X_train_B, y_train_B = B_train_ds.get_data()  # [n_epochs, 64, 72], [n_epochs]
X_test_B, y_test_B = B_test_ds.get_data()

FEATURE_LEN = X_train_B.shape[-1]  # 72 отсчёта
print(f'Electrode {ELECTRODE!r} -> index {ch_idx}; FEATURE_LEN={FEATURE_LEN}')
print(f'Train {tuple(X_train_B.shape)}, Test {tuple(X_test_B.shape)}')

In [ ]:
subjects = ['B']

# Штатное train/test-разбиение датасета: train -> обучение, test -> валидация.
data_train = {'B': standardize_per_sample(X_train_B[:, ch_idx, :].numpy())}
labels_train = {'B': y_train_B.numpy()}
data_val = {'B': standardize_per_sample(X_test_B[:, ch_idx, :].numpy())}
labels_val = {'B': y_test_B.numpy()}

for subj in subjects:
    # train перемешиваем, val оставляем как есть
    perm = torch.randperm(len(labels_train[subj]))
    data_train[subj] = torch.tensor(data_train[subj]).float()[perm]
    labels_train[subj] = torch.tensor(labels_train[subj].squeeze()).float()[perm]
    data_val[subj] = torch.tensor(data_val[subj]).float()
    labels_val[subj] = torch.tensor(labels_val[subj].squeeze()).float()

    print(f"{subj}: train {tuple(data_train[subj].shape)} "
          f"(pos={int((labels_train[subj]==1).sum())}), "
          f"val {tuple(data_val[subj].shape)} "
          f"(pos={int((labels_val[subj]==1).sum())})")

## Эксперименты для 5 каналов

In [ ]:
N_CHANNELS = 5

dataloaders = {}

for subj in subjects:
    train_data, train_labels = data_train[subj], labels_train[subj]
    val_data, val_labels = data_val[subj], labels_val[subj]

    train_data, train_labels = borderline_smote_torch(
        train_data, train_labels,
        k=15, m=10,
        standardize_for_neighbors=False
    )

    X_train, y_train = build_multichannel_subject_dataset_unique(
        train_data, train_labels, n_channels=N_CHANNELS
    )
    X_val, y_val = build_multichannel_subject_dataset_unique(
        val_data, val_labels, n_channels=N_CHANNELS
    )

    train_dataset = CNNMatrixDataset(tensors=(X_train, y_train), with_target=True)
    val_dataset = CNNMatrixDataset(tensors=(X_val, y_val), with_target=True)

    dataloaders[subj] = {
        'train': DataLoader(train_dataset, batch_size=1024, shuffle=True),
        'val': DataLoader(val_dataset, batch_size=1024, shuffle=True),
    }

In [ ]:
mc_dataloaders = {}
single_dataloaders = {}

for subj, dataloader in dataloaders.items():
    X_train, y_train = dataloader['train'].dataset.tensors
    X_val, y_val = dataloader['val'].dataset.tensors

    X_train_single = multichannel_to_single_channel(X_train)
    X_val_single = multichannel_to_single_channel(X_val)

    mc_train_ds = CNNMatrixDataset(tensors=(X_train, y_train), with_target=True)
    mc_val_ds = CNNMatrixDataset(tensors=(X_val, y_val), with_target=True)

    single_train_ds = CNNMatrixDataset(tensors=(X_train_single, y_train), with_target=True)
    single_val_ds = CNNMatrixDataset(tensors=(X_val_single, y_val), with_target=True)

    mc_dataloaders[subj] = {
        'train': DataLoader(mc_train_ds, batch_size=256, shuffle=True),
        'val': DataLoader(mc_val_ds, batch_size=256, shuffle=True)
    }

    single_dataloaders[subj] = {
        'train': DataLoader(single_train_ds, batch_size=256, shuffle=True),
        'val': DataLoader(single_val_ds, batch_size=256, shuffle=True)
    }

### EEGNet, 5 каналов (weight_decay=1e-5)

In [ ]:
criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

learning_params_mc = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

learning_params_single = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

res_eegnet_5ch = {}
single_models_eeg = {}
mc_models_eeg = {}

for subj in mc_dataloaders.keys():
    print(f'\n===== Subject {subj} =====')

    # ===== MULTI-CHANNEL =====
    mc_models_eeg[subj] = EEGNet(FEATURE_LEN, 5, F1=128, D=1, F2=256).to(my_device)
    loss_mc, acc_mc, _ = train_model(
        mc_models_eeg[subj], mc_dataloaders[subj], criterion, learning_params_mc, device=my_device
    )

    # ===== SINGLE-CHANNEL (MEAN) =====
    single_models_eeg[subj] = EEGNet(FEATURE_LEN, 1, F1=8, D=8, F2=8).to(my_device)
    loss_sc, acc_sc, _ = train_model(
        single_models_eeg[subj], single_dataloaders[subj], criterion, learning_params_single, device=my_device
    )

    print(f'MC — Acc: {acc_mc["Accuracy"][-1]:.3f}, ITR: {acc_mc["ITR"][-1]:.4f}, CI: [{acc_mc["Min Accuracy"][-1]:.3f}, {acc_mc["Max Accuracy"][-1]:.3f}]')
    print(f'SC — Acc: {acc_sc["Accuracy"][-1]:.3f}, ITR: {acc_sc["ITR"][-1]:.4f}, CI: [{acc_sc["Min Accuracy"][-1]:.3f}, {acc_sc["Max Accuracy"][-1]:.3f}]')

    res_eegnet_5ch[subj] = {
        'mc_accuracy': acc_mc['Accuracy'][-1],
        'mc_f1': acc_mc['F1-score'][-1],
        'mc_itr': acc_mc['ITR'][-1],
        'sc_accuracy': acc_sc['Accuracy'][-1],
        'sc_f1': acc_sc['F1-score'][-1],
        'sc_itr': acc_sc['ITR'][-1],
        'size': len(mc_dataloaders[subj]['val'].dataset),
        'mc_ci': (acc_mc['Min Accuracy'][-1], acc_mc['Max Accuracy'][-1]),
        'sc_ci': (acc_sc['Min Accuracy'][-1], acc_sc['Max Accuracy'][-1]),
    }

In [ ]:
rows = []
for subj, res in res_eegnet_5ch.items():
    rows.append({
        'Subject': subj,
        'Val size': res['size'],
        'MC Accuracy': res['mc_accuracy'],
        'MC F1': res['mc_f1'],
        'MC ITR': res['mc_itr'],
        'SC Accuracy': res['sc_accuracy'],
        'SC F1': res['sc_f1'],
        'SC ITR': res['sc_itr'],
    })

df = pd.DataFrame(rows).sort_values('Subject')
pd.set_option('display.precision', 3)
df

### BaseCNN, 5 каналов (weight_decay=1e-2)

In [ ]:
criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

learning_params_mc = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

learning_params_single = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

res_basecnn_5ch = {}
single_models_eeg = {}
mc_models_eeg = {}

for subj in mc_dataloaders.keys():
    print(f'\n===== Subject {subj} =====')

    # ===== MULTI-CHANNEL =====
    mc_models_eeg[subj] = BaseCNN(FEATURE_LEN, 5).to(my_device)
    loss_mc, acc_mc, _ = train_model(
        mc_models_eeg[subj], mc_dataloaders[subj], criterion, learning_params_mc, device=my_device
    )

    # ===== SINGLE-CHANNEL (MEAN) =====
    single_models_eeg[subj] = BaseCNN(FEATURE_LEN, 1).to(my_device)
    loss_sc, acc_sc, _ = train_model(
        single_models_eeg[subj], single_dataloaders[subj], criterion, learning_params_single, device=my_device
    )

    print(f'MC — Acc: {acc_mc["Accuracy"][-1]:.3f}, ITR: {acc_mc["ITR"][-1]:.4f}, CI: [{acc_mc["Min Accuracy"][-1]:.3f}, {acc_mc["Max Accuracy"][-1]:.3f}]')
    print(f'SC — Acc: {acc_sc["Accuracy"][-1]:.3f}, ITR: {acc_sc["ITR"][-1]:.4f}, CI: [{acc_sc["Min Accuracy"][-1]:.3f}, {acc_sc["Max Accuracy"][-1]:.3f}]')

    res_basecnn_5ch[subj] = {
        'mc_accuracy': acc_mc['Accuracy'][-1],
        'mc_f1': acc_mc['F1-score'][-1],
        'mc_itr': acc_mc['ITR'][-1],
        'sc_accuracy': acc_sc['Accuracy'][-1],
        'sc_f1': acc_sc['F1-score'][-1],
        'sc_itr': acc_sc['ITR'][-1],
        'size': len(mc_dataloaders[subj]['val'].dataset),
        'mc_ci': (acc_mc['Min Accuracy'][-1], acc_mc['Max Accuracy'][-1]),
        'sc_ci': (acc_sc['Min Accuracy'][-1], acc_sc['Max Accuracy'][-1]),
    }

In [ ]:
rows = []
for subj, res in res_basecnn_5ch.items():
    rows.append({
        'Subject': subj,
        'Val size': res['size'],
        'MC Accuracy': res['mc_accuracy'],
        'MC F1': res['mc_f1'],
        'MC ITR': res['mc_itr'],
        'SC Accuracy': res['sc_accuracy'],
        'SC F1': res['sc_f1'],
        'SC ITR': res['sc_itr'],
    })

df = pd.DataFrame(rows).sort_values('Subject')
pd.set_option('display.precision', 3)
df

## Эксперименты для 10 каналов

In [ ]:
N_CHANNELS = 10

dataloaders = {}

for subj in subjects:
    train_data, train_labels = data_train[subj], labels_train[subj]
    val_data, val_labels = data_val[subj], labels_val[subj]

    train_data, train_labels = borderline_smote_torch(
        train_data, train_labels,
        k=15, m=10,
        standardize_for_neighbors=False
    )

    X_train, y_train = build_multichannel_subject_dataset_unique(
        train_data, train_labels, n_channels=N_CHANNELS
    )
    X_val, y_val = build_multichannel_subject_dataset_unique(
        val_data, val_labels, n_channels=N_CHANNELS
    )

    train_dataset = CNNMatrixDataset(tensors=(X_train, y_train), with_target=True)
    val_dataset = CNNMatrixDataset(tensors=(X_val, y_val), with_target=True)

    dataloaders[subj] = {
        'train': DataLoader(train_dataset, batch_size=1024, shuffle=True),
        'val': DataLoader(val_dataset, batch_size=1024, shuffle=True),
    }

In [ ]:
mc_dataloaders = {}
single_dataloaders = {}

for subj, dataloader in dataloaders.items():
    X_train, y_train = dataloader['train'].dataset.tensors
    X_val, y_val = dataloader['val'].dataset.tensors

    X_train_single = multichannel_to_single_channel(X_train)
    X_val_single = multichannel_to_single_channel(X_val)

    mc_train_ds = CNNMatrixDataset(tensors=(X_train, y_train), with_target=True)
    mc_val_ds = CNNMatrixDataset(tensors=(X_val, y_val), with_target=True)

    single_train_ds = CNNMatrixDataset(tensors=(X_train_single, y_train), with_target=True)
    single_val_ds = CNNMatrixDataset(tensors=(X_val_single, y_val), with_target=True)

    mc_dataloaders[subj] = {
        'train': DataLoader(mc_train_ds, batch_size=256, shuffle=True),
        'val': DataLoader(mc_val_ds, batch_size=256, shuffle=True)
    }

    single_dataloaders[subj] = {
        'train': DataLoader(single_train_ds, batch_size=256, shuffle=True),
        'val': DataLoader(single_val_ds, batch_size=256, shuffle=True)
    }

### EEGNet, 10 каналов (weight_decay=1e-5)

In [ ]:
criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

learning_params_mc = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

learning_params_single = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

res_eegnet_10ch = {}
single_models_eeg = {}
mc_models_eeg = {}

for subj in mc_dataloaders.keys():
    print(f'\n===== Subject {subj} =====')

    # ===== MULTI-CHANNEL =====
    mc_models_eeg[subj] = EEGNet(FEATURE_LEN, 10, F1=128, D=1, F2=256).to(my_device)
    loss_mc, acc_mc, _ = train_model(
        mc_models_eeg[subj], mc_dataloaders[subj], criterion, learning_params_mc, device=my_device
    )

    # ===== SINGLE-CHANNEL (MEAN) =====
    single_models_eeg[subj] = EEGNet(FEATURE_LEN, 1, F1=8, D=8, F2=8).to(my_device)
    loss_sc, acc_sc, _ = train_model(
        single_models_eeg[subj], single_dataloaders[subj], criterion, learning_params_single, device=my_device
    )

    print(f'MC — Acc: {acc_mc["Accuracy"][-1]:.3f}, ITR: {acc_mc["ITR"][-1]:.4f}, CI: [{acc_mc["Min Accuracy"][-1]:.3f}, {acc_mc["Max Accuracy"][-1]:.3f}]')
    print(f'SC — Acc: {acc_sc["Accuracy"][-1]:.3f}, ITR: {acc_sc["ITR"][-1]:.4f}, CI: [{acc_sc["Min Accuracy"][-1]:.3f}, {acc_sc["Max Accuracy"][-1]:.3f}]')

    res_eegnet_10ch[subj] = {
        'mc_accuracy': acc_mc['Accuracy'][-1],
        'mc_f1': acc_mc['F1-score'][-1],
        'mc_itr': acc_mc['ITR'][-1],
        'sc_accuracy': acc_sc['Accuracy'][-1],
        'sc_f1': acc_sc['F1-score'][-1],
        'sc_itr': acc_sc['ITR'][-1],
        'size': len(mc_dataloaders[subj]['val'].dataset),
        'mc_ci': (acc_mc['Min Accuracy'][-1], acc_mc['Max Accuracy'][-1]),
        'sc_ci': (acc_sc['Min Accuracy'][-1], acc_sc['Max Accuracy'][-1]),
    }

In [ ]:
rows = []
for subj, res in res_eegnet_10ch.items():
    rows.append({
        'Subject': subj,
        'Val size': res['size'],
        'MC Accuracy': res['mc_accuracy'],
        'MC F1': res['mc_f1'],
        'MC ITR': res['mc_itr'],
        'SC Accuracy': res['sc_accuracy'],
        'SC F1': res['sc_f1'],
        'SC ITR': res['sc_itr'],
    })

df = pd.DataFrame(rows).sort_values('Subject')
pd.set_option('display.precision', 3)
df

### BaseCNN, 10 каналов (weight_decay=1e-2)

In [ ]:
criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

learning_params_mc = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

learning_params_single = {
    'num_epochs': 500,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'step_size': 5,
    'gamma': 1,
    'num_classes': 2,
    'model_type': 'CNN'
}

res_basecnn_10ch = {}
single_models_eeg = {}
mc_models_eeg = {}

for subj in mc_dataloaders.keys():
    print(f'\n===== Subject {subj} =====')

    # ===== MULTI-CHANNEL =====
    mc_models_eeg[subj] = BaseCNN(FEATURE_LEN, 10).to(my_device)
    loss_mc, acc_mc, _ = train_model(
        mc_models_eeg[subj], mc_dataloaders[subj], criterion, learning_params_mc, device=my_device
    )

    # ===== SINGLE-CHANNEL (MEAN) =====
    single_models_eeg[subj] = BaseCNN(FEATURE_LEN, 1).to(my_device)
    loss_sc, acc_sc, _ = train_model(
        single_models_eeg[subj], single_dataloaders[subj], criterion, learning_params_single, device=my_device
    )

    print(f'MC — Acc: {acc_mc["Accuracy"][-1]:.3f}, ITR: {acc_mc["ITR"][-1]:.4f}, CI: [{acc_mc["Min Accuracy"][-1]:.3f}, {acc_mc["Max Accuracy"][-1]:.3f}]')
    print(f'SC — Acc: {acc_sc["Accuracy"][-1]:.3f}, ITR: {acc_sc["ITR"][-1]:.4f}, CI: [{acc_sc["Min Accuracy"][-1]:.3f}, {acc_sc["Max Accuracy"][-1]:.3f}]')

    res_basecnn_10ch[subj] = {
        'mc_accuracy': acc_mc['Accuracy'][-1],
        'mc_f1': acc_mc['F1-score'][-1],
        'mc_itr': acc_mc['ITR'][-1],
        'sc_accuracy': acc_sc['Accuracy'][-1],
        'sc_f1': acc_sc['F1-score'][-1],
        'sc_itr': acc_sc['ITR'][-1],
        'size': len(mc_dataloaders[subj]['val'].dataset),
        'mc_ci': (acc_mc['Min Accuracy'][-1], acc_mc['Max Accuracy'][-1]),
        'sc_ci': (acc_sc['Min Accuracy'][-1], acc_sc['Max Accuracy'][-1]),
    }

In [ ]:
rows = []
for subj, res in res_basecnn_10ch.items():
    rows.append({
        'Subject': subj,
        'Val size': res['size'],
        'MC Accuracy': res['mc_accuracy'],
        'MC F1': res['mc_f1'],
        'MC ITR': res['mc_itr'],
        'SC Accuracy': res['sc_accuracy'],
        'SC F1': res['sc_f1'],
        'SC ITR': res['sc_itr'],
    })

df = pd.DataFrame(rows).sort_values('Subject')
pd.set_option('display.precision', 3)
df

## SVM с усреднением эпох

Группируем N эпох → усредняем → обучаем SVM на усреднённых данных (индивидуальная модель Subject B).

In [ ]:
from sklearn.svm import SVC

def compute_itr(accuracy, n_classes=2):
    """Wolpaw's Information Transfer Rate (bits/trial)."""
    P = float(accuracy)
    N = int(n_classes)
    if P <= 0 or P >= 1:
        P = np.clip(P, 1e-10, 1 - 1e-10)
    if N <= 1:
        return 0.0
    return float(np.log2(N) + P * np.log2(P) + (1 - P) * np.log2((1 - P) / (N - 1)))

def get_metrics(preds, labels, n_classes=2):
    corrects = (preds == labels).sum()
    min_acc, max_acc = proportion_confint(corrects, len(preds), 0.05)
    accuracy = corrects / len(preds)
    itr = compute_itr(accuracy, n_classes)
    return {'Accuracy': accuracy, 'ITR': itr, 'Min Accuracy': min_acc, 'Max Accuracy': max_acc}

In [ ]:
res_svm_epoch = {}

for N_CHANNELS in [5, 10]:
    print(f"=== N_CHANNELS = {N_CHANNELS} ===")

    for subj in subjects:
        train_data, train_labels = data_train[subj], labels_train[subj]
        val_data, val_labels = data_val[subj], labels_val[subj]

        train_data, train_labels = borderline_smote_torch(
            train_data, train_labels, k=15, m=10, standardize_for_neighbors=False
        )

        X_train, y_train = build_multichannel_subject_dataset_unique(
            train_data, train_labels, n_channels=N_CHANNELS
        )
        X_val, y_val = build_multichannel_subject_dataset_unique(
            val_data, val_labels, n_channels=N_CHANNELS
        )

        X_train_avg = X_train.mean(dim=1).numpy()
        y_train_avg = y_train.numpy()
        X_val_avg = X_val.mean(dim=1).numpy()
        y_val_avg = y_val.numpy()

        clf = SVC().fit(X_train_avg, y_train_avg)
        preds = clf.predict(X_val_avg)

        acc = get_metrics(preds, y_val_avg)
        print(f"{subj} — Accuracy: {acc['Accuracy']:.3f}, ITR: {acc['ITR']:.4f} bits/trial, "
              f"CI: [{acc['Min Accuracy']:.3f}, {acc['Max Accuracy']:.3f}]")

        res_svm_epoch[(N_CHANNELS, subj)] = {
            'accuracy': acc['Accuracy'],
            'itr': acc['ITR'],
            'size': len(y_val_avg),
            'lower_ci': acc['Min Accuracy'],
            'upper_ci': acc['Max Accuracy']
        }
    print()

In [ ]:
for N_CH in [5, 10]:
    rows = []
    for (n, subj), res in res_svm_epoch.items():
        if n != N_CH:
            continue
        rows.append({
            'Subject': subj,
            'Val size': res['size'],
            'Accuracy': res['accuracy'],
            'ITR': res['itr'],
            'CI low': res['lower_ci'],
            'CI high': res['upper_ci'],
        })

    df = pd.DataFrame(rows).sort_values('Subject')
    print(f'\n=== SVM Individual, N_CHANNELS = {N_CH} ===')
    display(df)

## Сводное сравнение (epoch averaging)

Сравнение MC vs SC для EEGNet / BaseCNN и SVM по обоим N (5 и 10) на Subject B.

In [ ]:
def _mean_mc_sc(res_dict, n_ch=None):
    vals = {}
    for key, res in res_dict.items():
        if n_ch is not None and isinstance(key, tuple) and key[0] != n_ch:
            continue
        for metric in ['mc_accuracy', 'mc_itr', 'sc_accuracy', 'sc_itr']:
            vals.setdefault(metric, []).append(res[metric])
    return {k: np.mean(v) for k, v in vals.items()}

def _mean_svm(res_dict, n_ch=None):
    vals = {}
    for key, res in res_dict.items():
        if n_ch is not None and isinstance(key, tuple) and key[0] != n_ch:
            continue
        for metric in ['accuracy', 'itr']:
            vals.setdefault(metric, []).append(res[metric])
    return {k: np.mean(v) for k, v in vals.items()}

rows = []

ea5_eeg = _mean_mc_sc(res_eegnet_5ch)
ea5_cnn = _mean_mc_sc(res_basecnn_5ch)
ea5_svm = _mean_svm(res_svm_epoch, n_ch=5)
rows.append({
    'Approach': 'Epoch Avg', 'Config': 'N=5',
    'EEGNet MC Acc': ea5_eeg['mc_accuracy'], 'EEGNet MC ITR': ea5_eeg['mc_itr'],
    'EEGNet SC Acc': ea5_eeg['sc_accuracy'], 'EEGNet SC ITR': ea5_eeg['sc_itr'],
    'BaseCNN MC Acc': ea5_cnn['mc_accuracy'], 'BaseCNN MC ITR': ea5_cnn['mc_itr'],
    'BaseCNN SC Acc': ea5_cnn['sc_accuracy'], 'BaseCNN SC ITR': ea5_cnn['sc_itr'],
    'SVM Acc': ea5_svm['accuracy'], 'SVM ITR': ea5_svm['itr'],
})

ea10_eeg = _mean_mc_sc(res_eegnet_10ch)
ea10_cnn = _mean_mc_sc(res_basecnn_10ch)
ea10_svm = _mean_svm(res_svm_epoch, n_ch=10)
rows.append({
    'Approach': 'Epoch Avg', 'Config': 'N=10',
    'EEGNet MC Acc': ea10_eeg['mc_accuracy'], 'EEGNet MC ITR': ea10_eeg['mc_itr'],
    'EEGNet SC Acc': ea10_eeg['sc_accuracy'], 'EEGNet SC ITR': ea10_eeg['sc_itr'],
    'BaseCNN MC Acc': ea10_cnn['mc_accuracy'], 'BaseCNN MC ITR': ea10_cnn['mc_itr'],
    'BaseCNN SC Acc': ea10_cnn['sc_accuracy'], 'BaseCNN SC ITR': ea10_cnn['sc_itr'],
    'SVM Acc': ea10_svm['accuracy'], 'SVM ITR': ea10_svm['itr'],
})

df_unified = pd.DataFrame(rows)
pd.set_option('display.precision', 3)
pd.set_option('display.max_columns', 20)
display(df_unified)

latex_table = df_unified.to_latex(
    index=False,
    float_format="%.3f",
    caption="Epoch averaging comparison (BCI Subject B, electrode Pz)",
    label="tab:epoch_avg_subjectB"
)
print(latex_table)

# Многоэлектродный вариант (все 64 канала)

Проверяем гипотезу из сравнения с `epoch_averaging.ipynb`: одиночный `Pz` —
вероятная причина слабого детектирования P300. Здесь повторяем ту же батарею
экспериментов (epoch averaging N=5, N=10 для EEGNet / BaseCNN / SVM, MC vs SC),
но вместо одного электрода используем **все 64 канала** монтажа.

Обобщение схемы MC/SC на несколько электродов (при `E=1` сводится к исходному
одноэлектродному поведению):

- эпоха теперь несёт `E` реальных электродов: `[T, E, 72]`;
- группировка `N` эпох одного класса → `[T', N, E, 72]`;
- **MC** = `[T', N·E, 72]` (эпохи × электроды как каналы модели);
- **SC** = усреднение по оси эпох → `[T', E, 72]` (стандартное многоэлектродное усреднение P300).

Гиперпараметры моделей и обучения — **те же, что в Pz-секции**, так что единственная
изменяемая переменная — число электродов. Итоговая таблица содержит обе строки
(Pz и all-64) для прямого сравнения.

In [ ]:
# --- Обобщённые помощники MC/SC для произвольного числа электродов E ---

def standardize_per_sample_me(X, eps=1e-8):
    """Z-score каждой пары (sample, electrode) по оси времени. X: [T, E, F]."""
    X = np.asarray(X, dtype=np.float32)
    mean = X.mean(axis=2, keepdims=True)
    std = X.std(axis=2, keepdims=True)
    return (X - mean) / np.maximum(std, eps)


def smote_multielectrode(X, y, **kw):
    """borderline_smote_torch работает с 2D; разворачиваем электроды+время и восстанавливаем форму."""
    if X.dim() == 3:                       # [T, E, F]
        T, E, F = X.shape
        X_aug, y_aug = borderline_smote_torch(X.reshape(T, E * F), y, **kw)
        return X_aug.reshape(-1, E, F), y_aug
    return borderline_smote_torch(X, y, **kw)


def to_mc(X):
    """Эпохи (и электроды) как каналы. [T',N,E,F]->[T',N*E,F]; [T',N,F] без изменений."""
    if X.dim() == 4:
        T, N, E, F = X.shape
        return X.reshape(T, N * E, F)
    return X


def to_sc(X):
    """Усреднение по оси эпох (dim=1), электроды сохраняются.
    [T',N,E,F]->[T',E,F]; [T',N,F]->[T',1,F]."""
    if X.dim() == 4:
        return X.mean(dim=1)
    return X.mean(dim=1, keepdim=True)


def make_mc_sc_loaders(data, labels, val_data, val_labels, n_avg, batch_size=256, smote=True):
    """Строит MC- и SC-даталоадеры для произвольного E. Возвращает (mc_dl, sc_dl, mc_ch, sc_ch)."""
    if smote:
        data, labels = smote_multielectrode(
            data, labels, k=15, m=10, standardize_for_neighbors=False
        )

    X_tr, y_tr = build_multichannel_subject_dataset_unique(data, labels, n_channels=n_avg)
    X_vl, y_vl = build_multichannel_subject_dataset_unique(val_data, val_labels, n_channels=n_avg)

    mc_tr, sc_tr = to_mc(X_tr), to_sc(X_tr)
    mc_vl, sc_vl = to_mc(X_vl), to_sc(X_vl)

    mc_dl = {
        'train': DataLoader(CNNMatrixDataset(tensors=(mc_tr, y_tr), with_target=True),
                            batch_size=batch_size, shuffle=True),
        'val': DataLoader(CNNMatrixDataset(tensors=(mc_vl, y_vl), with_target=True),
                          batch_size=batch_size, shuffle=True),
    }
    sc_dl = {
        'train': DataLoader(CNNMatrixDataset(tensors=(sc_tr, y_tr), with_target=True),
                            batch_size=batch_size, shuffle=True),
        'val': DataLoader(CNNMatrixDataset(tensors=(sc_vl, y_vl), with_target=True),
                          batch_size=batch_size, shuffle=True),
    }
    return mc_dl, sc_dl, mc_tr.shape[1], sc_tr.shape[1]


def _pack_mc_sc(acc_mc, acc_sc, val_size):
    return {
        'mc_accuracy': acc_mc['Accuracy'][-1], 'mc_f1': acc_mc['F1-score'][-1],
        'mc_itr': acc_mc['ITR'][-1],
        'sc_accuracy': acc_sc['Accuracy'][-1], 'sc_f1': acc_sc['F1-score'][-1],
        'sc_itr': acc_sc['ITR'][-1], 'size': val_size,
        'mc_ci': (acc_mc['Min Accuracy'][-1], acc_mc['Max Accuracy'][-1]),
        'sc_ci': (acc_sc['Min Accuracy'][-1], acc_sc['Max Accuracy'][-1]),
    }

In [ ]:
# --- Подготовка данных: все 64 электрода ---
# X_train_B / X_test_B уже загружены выше как [n_epochs, 64, 72].

E_ALL = X_train_B.shape[1]  # 64

data_train_me = {'B': standardize_per_sample_me(X_train_B.numpy())}   # [T, 64, 72]
labels_train_me = {'B': y_train_B.numpy()}
data_val_me = {'B': standardize_per_sample_me(X_test_B.numpy())}
labels_val_me = {'B': y_test_B.numpy()}

for subj in subjects:
    perm = torch.randperm(len(labels_train_me[subj]))
    data_train_me[subj] = torch.tensor(data_train_me[subj]).float()[perm]
    labels_train_me[subj] = torch.tensor(labels_train_me[subj].squeeze()).float()[perm]
    data_val_me[subj] = torch.tensor(data_val_me[subj]).float()
    labels_val_me[subj] = torch.tensor(labels_val_me[subj].squeeze()).float()

    print(f"{subj} [all {E_ALL} ch]: train {tuple(data_train_me[subj].shape)} "
          f"(pos={int((labels_train_me[subj]==1).sum())}), "
          f"val {tuple(data_val_me[subj].shape)} "
          f"(pos={int((labels_val_me[subj]==1).sum())})")

In [ ]:
# --- EEGNet и BaseCNN на всех 64 электродах, N=5 и N=10 ---
import gc

criterion = nn.MSELoss()
my_device = torch.device('cuda:0')

# Pz-секция оставляет обученные модели на GPU (mc_models_eeg / single_models_eeg)
# плюс кэш аллокатора PyTorch — освобождаем перед тяжёлой all-64 секцией.
for _nm in ['mc_models_eeg', 'single_models_eeg', 'mc_model', 'sc_model']:
    if _nm in globals():
        del globals()[_nm]
gc.collect()
torch.cuda.empty_cache()

def _fit_free(model, dataloaders, learning_params):
    """Обучает модель, возвращает метрики и сразу освобождает её с GPU."""
    _, acc, _ = train_model(model, dataloaders, criterion, learning_params, device=my_device)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return acc

def _lp(weight_decay):
    return {
        'num_epochs': 500, 'lr': 1e-4, 'weight_decay': weight_decay,
        'step_size': 5, 'gamma': 1, 'num_classes': 2, 'model_type': 'CNN',
    }

res_eegnet_me = {}    # ключ (N, subj)
res_basecnn_me = {}

for N in [5, 10]:
    for subj in subjects:
        # batch_size=32: MC на all-64 даёт N*64 каналов (до 640) — на T4 нужен малый батч.
        mc_dl, sc_dl, mc_ch, sc_ch = make_mc_sc_loaders(
            data_train_me[subj], labels_train_me[subj],
            data_val_me[subj], labels_val_me[subj], n_avg=N, batch_size=32,
        )
        val_size = len(sc_dl['val'].dataset)
        print(f'\n===== Subject {subj}, N={N}, MC channels={mc_ch}, SC channels={sc_ch} =====')

        # ---- EEGNet (те же гиперпараметры, что в Pz-секции) ----
        acc_mc = _fit_free(EEGNet(FEATURE_LEN, mc_ch, F1=128, D=1, F2=256).to(my_device), mc_dl, _lp(1e-5))
        acc_sc = _fit_free(EEGNet(FEATURE_LEN, sc_ch, F1=8, D=8, F2=8).to(my_device), sc_dl, _lp(1e-5))
        res_eegnet_me[(N, subj)] = _pack_mc_sc(acc_mc, acc_sc, val_size)
        print(f'EEGNet  — MC Acc: {acc_mc["Accuracy"][-1]:.3f} (ITR {acc_mc["ITR"][-1]:.4f}), '
              f'SC Acc: {acc_sc["Accuracy"][-1]:.3f} (ITR {acc_sc["ITR"][-1]:.4f})')

        # ---- BaseCNN ----
        acc_mc = _fit_free(BaseCNN(FEATURE_LEN, mc_ch).to(my_device), mc_dl, _lp(1e-2))
        acc_sc = _fit_free(BaseCNN(FEATURE_LEN, sc_ch).to(my_device), sc_dl, _lp(1e-2))
        res_basecnn_me[(N, subj)] = _pack_mc_sc(acc_mc, acc_sc, val_size)
        print(f'BaseCNN — MC Acc: {acc_mc["Accuracy"][-1]:.3f} (ITR {acc_mc["ITR"][-1]:.4f}), '
              f'SC Acc: {acc_sc["Accuracy"][-1]:.3f} (ITR {acc_sc["ITR"][-1]:.4f})')

In [ ]:
# --- SVM с усреднением эпох на всех 64 электродах ---
from sklearn.svm import SVC

res_svm_me = {}   # ключ (N, subj)

for N in [5, 10]:
    print(f"=== SVM (all {E_ALL} ch), N = {N} ===")
    for subj in subjects:
        tr, trl = smote_multielectrode(
            data_train_me[subj], labels_train_me[subj],
            k=15, m=10, standardize_for_neighbors=False,
        )
        X_tr, y_tr = build_multichannel_subject_dataset_unique(tr, trl, n_channels=N)
        X_vl, y_vl = build_multichannel_subject_dataset_unique(
            data_val_me[subj], labels_val_me[subj], n_channels=N
        )

        # усредняем по эпохам, сохраняем электроды, разворачиваем в вектор для SVM
        X_tr_avg = X_tr.mean(dim=1).reshape(X_tr.shape[0], -1).numpy()
        X_vl_avg = X_vl.mean(dim=1).reshape(X_vl.shape[0], -1).numpy()

        clf = SVC().fit(X_tr_avg, y_tr.numpy())
        preds = clf.predict(X_vl_avg)
        acc = get_metrics(preds, y_vl.numpy())

        res_svm_me[(N, subj)] = {
            'accuracy': acc['Accuracy'], 'itr': acc['ITR'], 'size': len(y_vl),
            'lower_ci': acc['Min Accuracy'], 'upper_ci': acc['Max Accuracy'],
        }
        print(f"{subj} — Accuracy: {acc['Accuracy']:.3f}, ITR: {acc['ITR']:.4f} bits/trial, "
              f"CI: [{acc['Min Accuracy']:.3f}, {acc['Max Accuracy']:.3f}]")
    print()

## Сводное сравнение: Pz против всех 64 электродов

Объединяем результаты одноэлектродной (Pz) и многоэлектродной (all-64) секций
в одну таблицу. Единственная изменяемая переменная — число электродов.

In [ ]:
# --- Сводная таблица Pz vs all-64 (Subject B) ---
pz_eeg = {5: res_eegnet_5ch, 10: res_eegnet_10ch}
pz_cnn = {5: res_basecnn_5ch, 10: res_basecnn_10ch}


def _row(label, config, e, c, s):
    return {
        'Electrodes': label, 'Config': config,
        'EEGNet MC Acc': e['mc_accuracy'], 'EEGNet MC ITR': e['mc_itr'],
        'EEGNet SC Acc': e['sc_accuracy'], 'EEGNet SC ITR': e['sc_itr'],
        'BaseCNN MC Acc': c['mc_accuracy'], 'BaseCNN MC ITR': c['mc_itr'],
        'BaseCNN SC Acc': c['sc_accuracy'], 'BaseCNN SC ITR': c['sc_itr'],
        'SVM Acc': s['accuracy'], 'SVM ITR': s['itr'],
    }

rows = []
for subj in subjects:
    for N in [5, 10]:
        rows.append(_row('Pz', f'N={N}',
                         pz_eeg[N][subj], pz_cnn[N][subj], res_svm_epoch[(N, subj)]))
        rows.append(_row(f'all {E_ALL}', f'N={N}',
                         res_eegnet_me[(N, subj)], res_basecnn_me[(N, subj)], res_svm_me[(N, subj)]))

df_elec = pd.DataFrame(rows)
pd.set_option('display.precision', 3)
pd.set_option('display.max_columns', 20)
display(df_elec)

latex_table = df_elec.to_latex(
    index=False,
    float_format="%.3f",
    caption="Electrode-count comparison (BCI Subject B): single Pz vs all 64 channels",
    label="tab:electrodes_subjectB",
)
print(latex_table)